# Similarity Engine Validation

This notebook checks data completeness, mathematical invariants, contextual plausibility, and ranking stability for the deployed feature index. The sensitivity section varies weights around the default and flags high-composite / low-topography matches. These are screening checks, not expert-labelled scientific validation.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / "app" / "data" / "sites.json").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate app/data/sites.json from the notebook working directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "app" / "data"
VALIDATION_DIR = DATA_DIR / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# These are the canonical feature vectors consumed by the runtime index.
sites_by_body = json.loads((DATA_DIR / "sites.json").read_text(encoding="utf-8"))
full_data = {
    site["site_id"]: site
    for body in ("moon", "mars")
    for site in sites_by_body.get(body, [])
}
# Keep provenance catalogs available for source review; feature validation uses sites.json.
moon_mars_provenance = json.loads((DATA_DIR / "moon_mars_full_real_data.json").read_text(encoding="utf-8"))
earth_provenance = json.loads((DATA_DIR / "earth_full_real_data.json").read_text(encoding="utf-8"))
taxonomy = json.loads((DATA_DIR / "taxonomy.json").read_text(encoding="utf-8"))
feature_stats = json.loads((DATA_DIR / "feature_stats.json").read_text(encoding="utf-8"))

TOPOGRAPHY_FEATURES = [item["id"] for item in feature_stats["topography_features"]]
LITHOLOGY_FEATURES = [item["id"] for item in taxonomy["lithology_classes"]]
MINERAL_FEATURES = [item["id"] for item in taxonomy["mineral_dimensions"]]
FLAG_FEATURES = ["ndvi", "is_impact", "has_ice", "is_volcanic"]

print(f"Loaded {len(full_data)} runtime Moon/Mars feature vectors from sites.json")
print(f"Loaded {len(moon_mars_provenance)} Moon/Mars provenance records and {len(earth_provenance)} Earth provenance records")
print(f"Validation outputs: {VALIDATION_DIR}")


## 1. Data contract and reusable range validation

The engine's target vector contains six topographic values, seven geology fractions, and nine mineral dimensions. Mineral dimensions may be missing and are handled by the engine's mask. Topography and geology are required for every target site.

In [ ]:
RANGE_RULES = {
    **{feature: (0.0, None) for feature in TOPOGRAPHY_FEATURES if feature != "hypsometric_integral"},
    "hypsometric_integral": (0.0, 1.0),
    **{feature: (0.0, 1.0) for feature in LITHOLOGY_FEATURES},
    **{feature: (0.0, 1.0) for feature in MINERAL_FEATURES},
    "ndvi": (0.0, 1.0),
}

def validate_vector_ranges(vector: dict, rules: dict[str, tuple[float | None, float | None]] = RANGE_RULES, allow_missing: bool = True) -> pd.DataFrame:
    """Validate numeric ranges for any Earth or planetary feature vector."""
    rows = []
    for feature, (lower, upper) in rules.items():
        value = vector.get(feature)
        missing = value is None or (isinstance(value, (float, np.floating)) and not np.isfinite(value))
        numeric = not missing and isinstance(value, (int, float, np.number))
        in_range = numeric and (lower is None or float(value) >= lower) and (upper is None or float(value) <= upper)
        rows.append({
            "feature": feature,
            "value": None if missing else float(value) if numeric else value,
            "missing": missing,
            "valid": bool((allow_missing and missing) or in_range),
            "lower_bound": lower,
            "upper_bound": upper,
            "failure_reason": "missing" if missing else None if in_range else "non-numeric or out-of-range",
        })
    return pd.DataFrame(rows)

def flatten_json_site(site: dict) -> dict:
    """Flatten the canonical runtime feature vector stored in sites.json."""
    topo = dict(zip(TOPOGRAPHY_FEATURES, site.get("topo", [])))
    geology = dict(zip(LITHOLOGY_FEATURES, site.get("geo", [])))
    minerals = dict(zip(MINERAL_FEATURES, site.get("min", [])))
    return {
        "site_id": site.get("site_id"), "body": site.get("body"), "name": site.get("name"),
        "lat": site.get("lat"), "lon": site.get("lon"),
        **topo, **geology,
        **{key: (None if value is None or float(value) < 0 else float(value)) for key, value in minerals.items()},
        **site.get("flags", {}),
    }

print("Range validator ready for canonical runtime vectors.")


In [ ]:
site_rows = []
range_rows = []
for site_id, site in full_data.items():
    vector = flatten_json_site(site)
    checked = validate_vector_ranges(vector)
    checked.insert(0, "site_id", site_id)
    range_rows.append(checked)
    site_rows.append({
        "site_id": site_id,
        "body": site["body"],
        "name": site["name"],
        "topography_present": sum(vector.get(f) is not None for f in TOPOGRAPHY_FEATURES),
        "geology_present": sum(vector.get(f) is not None for f in LITHOLOGY_FEATURES),
        "mineralogy_present": sum(vector.get(f) is not None for f in MINERAL_FEATURES),
        "flags_present": sum(vector.get(f) is not None for f in FLAG_FEATURES),
        "topography_missing": ", ".join(f for f in TOPOGRAPHY_FEATURES if vector.get(f) is None),
        "geology_missing": ", ".join(f for f in LITHOLOGY_FEATURES if vector.get(f) is None),
        "mineralogy_missing": ", ".join(f for f in MINERAL_FEATURES if vector.get(f) is None),
        "flags_missing": ", ".join(f for f in FLAG_FEATURES if vector.get(f) is None),
    })

site_presence = pd.DataFrame(site_rows)
range_results = pd.concat(range_rows, ignore_index=True)
topography_frame = pd.DataFrame([
    {feature: flatten_json_site(site).get(feature) for feature in TOPOGRAPHY_FEATURES}
    | {"site_id": site_id, "body": site["body"]}
    for site_id, site in full_data.items()
])
descriptive_stats = topography_frame[TOPOGRAPHY_FEATURES].describe().T.reset_index(names="feature")
descriptive_stats["missing"] = topography_frame[TOPOGRAPHY_FEATURES].isna().sum().to_numpy()

display(site_presence)
display(descriptive_stats)


## 2. Mathematical validation

These checks validate invariants rather than scientific truth: valid ranges, identity behaviour, monotonic degradation, and bounded scores.

In [ ]:
from app.services.feature_index import FeatureIndexStore

store = FeatureIndexStore(DATA_DIR)
lithology_matrix = store.lithology_matrix

def cosine_similarity(left: np.ndarray, right: np.ndarray) -> float:
    denominator = np.linalg.norm(left) * np.linalg.norm(right)
    return float(np.dot(left, right) / denominator) if denominator > 0 else 0.5

def topography_score(left: np.ndarray, right: np.ndarray) -> float:
    z_left = (left - store.topo_medians) / store.topo_iqrs
    z_right = (right - store.topo_medians) / store.topo_iqrs
    return float(np.exp(-np.mean((z_left - z_right) ** 2) / store.sigma_topo**2))

def geology_score(left: np.ndarray, right: np.ndarray) -> float:
    cross = float(left @ lithology_matrix @ right)
    self_left = float(left @ lithology_matrix @ left)
    self_right = float(right @ lithology_matrix @ right)
    denominator = np.sqrt(max(self_left * self_right, 1e-8))
    return float(np.clip(cross / denominator, 0.0, 1.0))

math_rows = [
    {"family": "geology", "test": "similarity_matrix_symmetric", "passed": bool(np.allclose(lithology_matrix, lithology_matrix.T))},
    {"family": "geology", "test": "similarity_matrix_bounded", "passed": bool(np.all((lithology_matrix >= 0) & (lithology_matrix <= 1)))},
    {"family": "geology", "test": "similarity_matrix_diagonal_is_one", "passed": bool(np.allclose(np.diag(lithology_matrix), 1.0))},
]
for body in ("moon", "mars", "earth"):
    index = store.indices[body]
    math_rows.extend([
        {"family": "topography", "test": f"{body}_topography_finite", "passed": bool(np.isfinite(index.topo).all())},
        {"family": "geology", "test": f"{body}_geology_finite", "passed": bool(np.isfinite(index.geo).all())},
        {"family": "geology", "test": f"{body}_geology_sums_to_one", "passed": bool(np.allclose(index.geo.sum(axis=1), 1.0, atol=0.02))},
        {"family": "mineralogy", "test": f"{body}_minerals_bounded", "passed": bool(np.all((index.min_vec[index.min_mask] >= 0) & (index.min_vec[index.min_mask] <= 1)))},
    ])

sample_topo = store.indices["moon"].topo[0]
same_geo = store.indices["moon"].geo[0]
same_min = store.indices["moon"].min_vec[0]
same_min = same_min[same_min >= 0]
math_rows.extend([
    {"family": "topography", "test": "identity_score_is_one", "passed": topography_score(sample_topo, sample_topo) > 0.999999},
    {"family": "geology", "test": "identity_score_is_one", "passed": geology_score(same_geo, same_geo) > 0.999999},
    {"family": "mineralogy", "test": "identity_score_is_one", "passed": cosine_similarity(same_min, same_min) > 0.999999},
])
perturbed = sample_topo.copy()
perturbed[0] += store.topo_iqrs[0] * 0.25
strongly_perturbed = sample_topo.copy()
strongly_perturbed[0] += store.topo_iqrs[0] * 2.0
math_rows.append({
    "family": "topography",
    "test": "score_decreases_with_distance",
    "passed": topography_score(sample_topo, perturbed) > topography_score(sample_topo, strongly_perturbed),
})
mathematical_results = pd.DataFrame(math_rows)
display(mathematical_results)

## 3. Fast contextual validation

Contextual checks are lightweight plausibility gates, not expert ground truth. A target passes a family check when the component score is available and bounded. Mineralogy also reports shared dimensions because a high score with little overlap has low confidence.

In [9]:
from app.schemas.analog import AnalogSearchRequest, SimilarityWeights
from app.services.similarity_engine import SimilarityEngine

engine = SimilarityEngine(store)

def contextual_validation(target_body: str, limit: int = 5) -> pd.DataFrame:
    rows = []
    for target in store.list_curated_sites(target_body):
        response = engine.search_analogs(AnalogSearchRequest(
            target_body=target_body,
            target_site_id=target.site_id,
            weights=SimilarityWeights(),
            k=limit,
        ))
        if not response.matches:
            rows.append({"target_site_id": target.site_id, "body": target_body, "passed": False, "reason": "no Earth matches"})
            continue
        scores = response.matches[0].scores
        shared = len(scores.compared_mineral_dimensions)
        rows.append({
            "target_site_id": target.site_id,
            "body": target_body,
            "topography_score": scores.topography_score,
            "geology_score": scores.geology_score,
            "mineralogy_score": scores.mineralogy_score,
            "shared_minerals": shared,
            "topography_context_pass": 0.0 <= scores.topography_score <= 1.0,
            "geology_context_pass": 0.0 <= scores.geology_score <= 1.0,
            "mineralogy_context_pass": 0.0 <= scores.mineralogy_score <= 1.0 and shared >= 1,
            "mineralogy_confidence": "high" if shared >= 4 else "low",
            "passed": 0.0 <= scores.topography_score <= 1.0 and 0.0 <= scores.geology_score <= 1.0,
        })
    return pd.DataFrame(rows)

contextual_results = pd.concat([contextual_validation("moon"), contextual_validation("mars")], ignore_index=True)
display(contextual_results)

,target_site_id,body,topography_score,geology_score,mineralogy_score,shared_minerals,topography_context_pass,geology_context_pass,mineralogy_context_pass,mineralogy_confidence,passed
0,moon_shackleton_rim,moon,0.0000,0.9188,0.9982,6,True,True,True,high,True
1,moon_marius_hills_pit,moon,0.9468,0.9737,0.8996,6,True,True,True,high,True
2,moon_aristarchus_plateau,moon,0.8374,0.8812,0.9740,6,True,True,True,high,True
3,moon_de_gerlache_rim,moon,0.0349,0.9078,0.9971,6,True,True,True,high,True
4,moon_tranquillitatis_pit,moon,0.9690,0.9946,0.8636,6,True,True,True,high,True
5,moon_schrodinger_basin,moon,0.7810,0.8728,0.8359,6,True,True,True,high,True
6,mars_jezero_crater,mars,0.8653,0.9620,0.9639,9,True,True,True,high,True
7,mars_arcadia_planitia,mars,0.9589,0.8879,0.9868,9,True,True,True,high,True
8,mars_elysium_planitia,mars,0.9760,0.9942,0.9637,9,True,True,True,high,True
9,mars_mawrth_vallis,mars,0.9140,0.9817,0.8850,9,True,True,True,high,True


## 4. Weight sensitivity and ranking stability

Compare the default weights with small ±0.05 and ±0.10 shifts to one family at a time. Each shift is offset proportionally across the other two weights so the weights still sum to one. For every target, report Top-1 changes, Top-5 overlap, and the best match's composite score. A separate flag calls attention to high-composite matches with weak topographic similarity; adjust the thresholds below to fit the review policy.


In [8]:
from app.schemas.analog import AnalogSearchRequest, SimilarityWeights
from app.services.similarity_engine import SimilarityEngine

engine = SimilarityEngine(store)
DEFAULT_WEIGHTS = {"topography": 0.45, "geology": 0.35, "mineralogy": 0.20}
SENSITIVITY_DELTAS = (-0.10, -0.05, 0.05, 0.10)
HIGH_COMPOSITE_THRESHOLD = 0.80
LOW_TOPOGRAPHY_THRESHOLD = 0.50

# Retain the broad scenarios and add local perturbations around the default.
RANKING_CONFIGS = {
    "default": DEFAULT_WEIGHTS.copy(),
    "topography_dominant": {"topography": 0.60, "geology": 0.25, "mineralogy": 0.15},
    "balanced": {"topography": 0.33, "geology": 0.33, "mineralogy": 0.34},
}
for family in DEFAULT_WEIGHTS:
    others = [key for key in DEFAULT_WEIGHTS if key != family]
    for delta in SENSITIVITY_DELTAS:
        config = DEFAULT_WEIGHTS.copy()
        config[family] += delta
        remaining = 1.0 - config[family]
        original_other_total = sum(DEFAULT_WEIGHTS[key] for key in others)
        for key in others:
            config[key] = remaining * DEFAULT_WEIGHTS[key] / original_other_total
        RANKING_CONFIGS[f"{family}_{delta:+.2f}"] = config

def ranking_stability() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rankings = []
    stability = []
    discordance = []
    for body in ("moon", "mars"):
        for target in store.list_curated_sites(body):
            config_results = {}
            for name, weights in RANKING_CONFIGS.items():
                response = engine.search_analogs(AnalogSearchRequest(
                    target_body=body,
                    target_site_id=target.site_id,
                    weights=SimilarityWeights(**weights),
                    k=5,
                ))
                matches = response.matches
                ids = [match.site.site_id for match in matches]
                config_results[name] = ids
                for rank, match in enumerate(matches, start=1):
                    rankings.append({
                        "target_site_id": target.site_id, "body": body,
                        "configuration": name, "rank": rank,
                        "earth_site_id": match.site.site_id,
                        "overall_score": match.scores.overall_score,
                        "topography_score": match.scores.topography_score,
                        "geology_score": match.scores.geology_score,
                        "mineralogy_score": match.scores.mineralogy_score,
                    })
            baseline_ids = config_results["default"]
            baseline_top5 = set(baseline_ids)
            baseline_top1 = baseline_ids[0] if baseline_ids else None
            target_stability = []
            for name, ids in config_results.items():
                overlap = len(baseline_top5.intersection(ids)) / max(len(baseline_top5), 1)
                row = {
                    "target_site_id": target.site_id, "body": body,
                    "configuration": name,
                    "top5_overlap_with_default": overlap,
                    "top1_changed_from_default": bool(ids and baseline_top1 and ids[0] != baseline_top1),
                    "first_result": ids[0] if ids else None,
                    "has_results": bool(ids),
                }
                stability.append(row)
                if name != "default":
                    target_stability.append(row)

            # The default first result is the candidate whose component scores are reviewed.
            default_response = engine.search_analogs(AnalogSearchRequest(
                target_body=body, target_site_id=target.site_id,
                weights=SimilarityWeights(**DEFAULT_WEIGHTS), k=1,
            ))
            if default_response.matches:
                best = default_response.matches[0]
                scores = best.scores
                discordance.append({
                    "target_site_id": target.site_id, "body": body,
                    "earth_site_id": best.site.site_id,
                    "overall_score": scores.overall_score,
                    "topography_score": scores.topography_score,
                    "geology_score": scores.geology_score,
                    "mineralogy_score": scores.mineralogy_score,
                    "shared_minerals": len(scores.compared_mineral_dimensions),
                    "review_low_topography": bool(scores.topography_score < LOW_TOPOGRAPHY_THRESHOLD),
                    "high_overall_low_topography": bool(
                        scores.overall_score >= HIGH_COMPOSITE_THRESHOLD
                        and scores.topography_score < LOW_TOPOGRAPHY_THRESHOLD
                    ),
                    "low_topography_high_mineralogy": bool(
                        scores.topography_score < LOW_TOPOGRAPHY_THRESHOLD
                        and scores.mineralogy_score >= HIGH_COMPOSITE_THRESHOLD
                    ),
                    "mean_top5_overlap_under_perturbation": float(np.mean([
                        row["top5_overlap_with_default"] for row in target_stability
                    ])) if target_stability else 1.0,
                    "top1_change_rate_under_perturbation": float(np.mean([
                        row["top1_changed_from_default"] for row in target_stability
                    ])) if target_stability else 0.0,
                })

    rankings_df = pd.DataFrame(rankings)
    stability_df = pd.DataFrame(stability)
    discordance_df = pd.DataFrame(discordance)
    local = stability_df[stability_df["configuration"].str.contains(r"_(?:\+|-)", regex=True)].copy()
    sensitivity_summary = local.groupby(["body", "configuration"], as_index=False).agg(
        mean_top5_overlap=("top5_overlap_with_default", "mean"),
        top1_change_rate=("top1_changed_from_default", "mean"),
    )
    # Aggregate by family and signed delta for a compact comparison table.
    sensitivity_summary["family"] = sensitivity_summary["configuration"].str.extract(r"^(topography|geology|mineralogy)_")[0]
    sensitivity_summary["delta"] = sensitivity_summary["configuration"].str.extract(r"_([+-]\d+\.\d+)$")[0].astype(float)
    sensitivity_summary = sensitivity_summary.sort_values(["body", "family", "delta"]).reset_index(drop=True)
    return rankings_df, stability_df, sensitivity_summary, discordance_df

rankings, ranking_stability_results, sensitivity_summary, discordance_results = ranking_stability()
display(sensitivity_summary)
display(discordance_results[discordance_results["review_low_topography"]])


,body,configuration,mean_top5_overlap,top1_change_rate,family,delta
0,mars,geology_-0.10,1.000000,0.000000,geology,-0.10
1,mars,geology_-0.05,1.000000,0.000000,geology,-0.05
2,mars,geology_+0.05,1.000000,0.000000,geology,0.05
3,mars,geology_+0.10,0.933333,0.000000,geology,0.10
4,mars,mineralogy_-0.10,1.000000,0.166667,mineralogy,-0.10
5,mars,mineralogy_-0.05,1.000000,0.000000,mineralogy,-0.05
6,mars,mineralogy_+0.05,0.933333,0.000000,mineralogy,0.05
7,mars,mineralogy_+0.10,0.933333,0.166667,mineralogy,0.10
8,mars,topography_-0.10,0.900000,0.000000,topography,-0.10
9,mars,topography_-0.05,0.933333,0.000000,topography,-0.05


,target_site_id,body,earth_site_id,overall_score,topography_score,geology_score,mineralogy_score,shared_minerals,review_low_topography,high_overall_low_topography,low_topography_high_mineralogy,mean_top5_overlap_under_perturbation,top1_change_rate_under_perturbation
0,moon_shackleton_rim,moon,earth_queen_maud_anorthosite,0.5212,0.0000,0.9188,0.9982,6,True,False,True,1.000000,0.142857
3,moon_de_gerlache_rim,moon,earth_queen_maud_anorthosite,0.5328,0.0349,0.9078,0.9971,6,True,False,True,0.971429,0.000000


In [ ]:
outputs = {
    "site_presence": site_presence,
    "range_results": range_results,
    "descriptive_stats": descriptive_stats,
    "mathematical_results": mathematical_results,
    "contextual_results": contextual_results,
    "ranking_results": rankings,
    "ranking_stability": ranking_stability_results,
    "weight_sensitivity_summary": sensitivity_summary,
    "score_discordance_review": discordance_results,
}
for name, frame in outputs.items():
    frame.to_csv(VALIDATION_DIR / f"{name}.csv", index=False)

local_stability = ranking_stability_results[
    ranking_stability_results["configuration"].str.contains(r"_(?:\+|-)", regex=True)
]
summary = {
    "site_count": int(len(site_presence)),
    "mathematical_pass_rate": float(mathematical_results["passed"].mean()),
    "contextual_pass_rate": float(contextual_results["passed"].mean()),
    "mean_non_default_top5_overlap": float(ranking_stability_results.loc[ranking_stability_results["configuration"] != "default", "top5_overlap_with_default"].mean()),
    "mean_local_perturbation_top5_overlap": float(local_stability["top5_overlap_with_default"].mean()),
    "local_perturbation_top1_change_rate": float(local_stability["top1_changed_from_default"].mean()),
    "high_composite_threshold": HIGH_COMPOSITE_THRESHOLD,
    "low_topography_threshold": LOW_TOPOGRAPHY_THRESHOLD,
    "high_overall_low_topography_count": int(discordance_results["high_overall_low_topography"].sum()),
    "low_topography_review_count": int(discordance_results["review_low_topography"].sum()),
    "scope": "Fast hackathon validation; sensitivity is ranking robustness, not expert-labelled scientific validation",
}
(VALIDATION_DIR / "validation_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
summary


## Interpretation guide

- A mathematical failure indicates a bug, invalid data contract, or invalid parameterization.
- A contextual failure is a plausibility warning, not proof that the method is scientifically wrong.
- Mineralogy with fewer than four shared dimensions is reported with low confidence.
- Ranking stability above 0.60 Top-5 overlap is a practical hackathon gate, not a scientific acceptance threshold.
- All CSV and JSON outputs are written under `app/data/validation/` for review and reuse.